In [1]:
# ============================================
# COMPLETE IMAGE CLASSIFICATION MODEL TRAINING
# FOR FASTAPI DEPLOYMENT
# ============================================

# Install required packages
!pip install -q datasets transformers evaluate pandas scikit-learn

# Import all required libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)
import evaluate
import torch
import pickle

# ============================================
# 1. LOAD AND PREPARE DATASET
# ============================================

# Load dataset from Hugging Face
df = pd.read_parquet("hf://datasets/ButterChicken98/plantvillage-image-text-pairs/data/train-00000-of-00001.parquet")

# Drop image column (we only need text data)
df = df.drop(columns=["image"])

# Explode captions to create individual rows
df = df.explode("captions")

# Save to CSV for reference
df.to_csv("dataset.csv", index=False)

print(f"Dataset shape: {df.shape}")
print(f"Sample data:\n{df.head()}")

# ============================================
# 2. ENCODE LABELS
# ============================================

# Define data columns
num_labels = 15
data_col_name = "captions"
label_col_name = "caption"

# Encode labels
encoder = LabelEncoder()
encoder.fit(df[label_col_name].tolist())
df["label"] = encoder.transform(df[label_col_name].tolist())

print(f"Number of unique labels: {len(encoder.classes_)}")
print(f"Label classes: {encoder.classes_}")

# ============================================
# 3. SPLIT DATASET
# ============================================

df_train, df_test = train_test_split(df, train_size=0.8, random_state=42)

# Convert to Hugging Face Dataset format
train_dataset = Dataset.from_pandas(df_train)
test_dataset = Dataset.from_pandas(df_test)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

# ============================================
# 4. TOKENIZATION
# ============================================

model_name = "distilbert-base-uncased"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tokenization function
def tokenize_fn(data):
    return tokenizer(data[data_col_name], truncation=True)

# Tokenize datasets
tokenized_train = train_dataset.map(tokenize_fn, batched=True)
tokenized_test = test_dataset.map(tokenize_fn, batched=True)

# Data collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# ============================================
# 5. MODEL INITIALIZATION
# ============================================

# Load pre-trained model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

# ============================================
# 6. EVALUATION METRICS
# ============================================

eval_metrics = evaluate.load("accuracy")

def metrics(eval_pred):
    logits, labels = eval_pred
    prediction = np.argmax(logits, axis=1)
    return eval_metrics.compute(predictions=prediction, references=labels)

# ============================================
# 7. TRAINING CONFIGURATION
# ============================================

training_arguments = TrainingArguments(
    output_dir="./checkpoints",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    learning_rate=0.00005,
    save_strategy="epoch",
    logging_strategy="epoch",
    eval_strategy="epoch",
    save_total_limit=2,
    weight_decay=0.01,
    report_to="none",
    load_best_model_at_end=True
)

# ============================================
# 8. TRAINER INITIALIZATION AND TRAINING
# ============================================

trainer = Trainer(
    model=model,
    args=training_arguments,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=metrics
)

# Train the model
print("Starting training...")
trainer.train()

# ============================================
# 9. SAVE THE MODEL AND TOKENIZER
# ============================================

# Save the trained model
model_save_path = "./plant_disease_model"
model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)

# Save the label encoder
with open(f"{model_save_path}/label_encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)

print(f"\n{'='*50}")
print("MODEL SAVED SUCCESSFULLY!")
print(f"{'='*50}")
print(f"Model location: {model_save_path}")
print(f"Files saved:")
print("  - config.json")
print("  - pytorch_model.bin")
print("  - tokenizer_config.json")
print("  - vocab.txt")
print("  - label_encoder.pkl")
print(f"{'='*50}")

# ============================================
# 10. EVALUATION
# ============================================

# Evaluate on test set
print("\nEvaluating model on test set...")
eval_results = trainer.evaluate()
print(f"Test accuracy: {eval_results['eval_accuracy']:.4f}")

print("\nTraining complete! Model is ready for FastAPI deployment.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Dataset shape: (82552, 2)
Sample data:
              caption                                           captions
0      Tomato healthy  A vibrant green and healthy tomato leaf with s...
0      Tomato healthy  A healthy Solanum lycopersicum leaf, free of d...
0      Tomato healthy  A fresh tomato leaf outdoors, glowing in sunli...
0      Tomato healthy  A clean and healthy tomato leaf image, perfect...
1  Tomato Late blight  A tomato leaf showing dark brown lesions and w...
Number of unique labels: 15
Label classes: ['Pepper bell Bacterial spot' 'Pepper bell healthy' 'Potato Early blight'
 'Potato Late blight' 'Potato healthy' 'Tomato Bacterial spot'
 'Tomato Early blight' 'Tomato Late blight' 'Tomato Leaf Mold'
 'Tomato Septoria leaf spot' 'Tomato Spider mites Two spotted spider mite'
 'Tomato Target Spot' 'Tomato YellowLeaf Curl Virus' 'Tomato healthy'
 'Tomato mosaic virus']
Train dataset size: 66041
Test dataset size: 16511


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

...

/tmp/ipython-input-339755114.py:139: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Starting training...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.022500,0.000002,1.000000
2,0.000000,0.000000,1.000000



MODEL SAVED SUCCESSFULLY!
Model location: ./plant_disease_model
Files saved:
  - config.json
  - pytorch_model.bin
  - tokenizer_config.json
  - vocab.txt
  - label_encoder.pkl

Evaluating model on test set...


Test accuracy: 1.0000

Training complete! Model is ready for FastAPI deployment.


In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import pickle

# Load model and tokenizer
model = AutoModelForSequenceClassification.from_pretrained("./plant_disease_model")
tokenizer = AutoTokenizer.from_pretrained("./plant_disease_model")

# Load label encoder
with open("./plant_disease_model/label_encoder.pkl", "rb") as f:
    encoder = pickle.load(f)


In [3]:
# Install required packages
!pip install -q fastapi uvicorn nest-asyncio pyngrok transformers torch


In [4]:
# ============================================
# FASTAPI FOR GOOGLE COLAB
# ============================================

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import pickle
import nest_asyncio
from pyngrok import ngrok
import uvicorn

# Apply nest_asyncio for Colab compatibility
nest_asyncio.apply()

# ============================================
# LOAD MODEL
# ============================================

print("Loading model and tokenizer...")

model_path = "./plant_disease_model"

# Load model, tokenizer, and label encoder
model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

with open(f"{model_path}/label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)

# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

print(f"✅ Model loaded on {device}")
print(f"📊 Number of classes: {len(label_encoder.classes_)}")

# ============================================
# CREATE FASTAPI APP
# ============================================

app = FastAPI(
    title="Plant Disease Classification API",
    description="Classify plant diseases from text descriptions",
    version="1.0.0"
)

# Define request/response models
class PredictionRequest(BaseModel):
    text: str

class PredictionResponse(BaseModel):
    predicted_class: str
    confidence: float
    all_predictions: dict

# ============================================
# API ENDPOINTS
# ============================================

@app.get("/")
async def root():
    """Root endpoint"""
    return {
        "message": "Plant Disease Classification API",
        "status": "running",
        "num_classes": len(label_encoder.classes_)
    }

@app.get("/health")
async def health_check():
    """Health check"""
    return {"status": "healthy", "model_loaded": True}

@app.post("/predict", response_model=PredictionResponse)
async def predict(request: PredictionRequest):
    """Make prediction from text"""
    try:
        # Tokenize
        inputs = tokenizer(
            request.text,
            truncation=True,
            padding=True,
            return_tensors="pt"
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # Predict
        with torch.no_grad():
            outputs = model(**inputs)
            probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
            confidence, predicted_idx = torch.max(probabilities, dim=1)

        # Decode
        predicted_class = label_encoder.inverse_transform([predicted_idx.item()])[0]

        # All predictions
        all_probs = probabilities[0].cpu().numpy()
        all_predictions = {
            label_encoder.classes_[i]: float(all_probs[i])
            for i in range(len(label_encoder.classes_))
        }
        all_predictions = dict(sorted(all_predictions.items(), key=lambda x: x[1], reverse=True))

        return PredictionResponse(
            predicted_class=predicted_class,
            confidence=float(confidence.item()),
            all_predictions=all_predictions
        )

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/classes")
async def get_classes():
    """Get all classes"""
    return {
        "classes": label_encoder.classes_.tolist(),
        "num_classes": len(label_encoder.classes_)
    }

# ============================================
# SETUP NGROK AND RUN SERVER
# ============================================

# Set ngrok auth token (✅ your token is added here)
ngrok.set_auth_token("34T2QFLGNHYYiQz2T0P9QIEWQ2M_6oGMapkZKhK3oer1UEiJ")

# Create public URL
public_url = ngrok.connect(8000)

print("\n" + "="*70)
print("🚀 FASTAPI SERVER IS RUNNING!")
print("="*70)
print(f"🌐 Public URL: {public_url}")
print(f"🏠 Local URL: http://127.0.0.1:8000")
print(f"📖 Interactive Docs: {public_url}/docs")
print(f"📋 Alternative Docs: {public_url}/redoc")
print("="*70)
print("\n✨ You can now make API requests to your public URL!")
print("\n💡 Example using curl:")
print(f'curl -X POST "{public_url}/predict" \\')
print('     -H "Content-Type: application/json" \\')
print('     -d \'{"text": "A tomato leaf with dark spots and lesions"}\'')
print("="*70 + "\n")

# Run server
# Run server (Colab-compatible)
import threading

def run_uvicorn():
    uvicorn.run(app, host="0.0.0.0", port=8000)

# Run in a separate thread
thread = threading.Thread(target=run_uvicorn, daemon=True)
thread.start()



Loading model and tokenizer...
✅ Model loaded on cuda
📊 Number of classes: 15

🚀 FASTAPI SERVER IS RUNNING!
🌐 Public URL: NgrokTunnel: "https://overtediously-uncravatted-ahmed.ngrok-free.dev" -> "http://localhost:8000"
🏠 Local URL: http://127.0.0.1:8000
📖 Interactive Docs: NgrokTunnel: "https://overtediously-uncravatted-ahmed.ngrok-free.dev" -> "http://localhost:8000"/docs
📋 Alternative Docs: NgrokTunnel: "https://overtediously-uncravatted-ahmed.ngrok-free.dev" -> "http://localhost:8000"/redoc

✨ You can now make API requests to your public URL!

💡 Example using curl:
curl -X POST "NgrokTunnel: "https://overtediously-uncravatted-ahmed.ngrok-free.dev" -> "http://localhost:8000"/predict" \
     -H "Content-Type: application/json" \
     -d '{"text": "A tomato leaf with dark spots and lesions"}'



In [5]:
import requests
import time
from pyngrok import ngrok # Import ngrok

# Add a small delay to allow the server to start
time.sleep(5) # Adjust the delay as needed

# Check if ngrok tunnel is active and get the public URL
tunnels = ngrok.get_tunnels()
public_url = None
for tunnel in tunnels:
    if tunnel.proto == "https":
        public_url = tunnel.public_url
        break

if public_url is None:
    print("Error: Ngrok tunnel not active. Please run the FastAPI cell again.")
else:
    API_URL = public_url
    print(f"Using Ngrok Public URL: {API_URL}")

    # Test 1: Health check
    try:
        response = requests.get(f"{API_URL}/health-check")
        print("Health Check:", response.json())
    except requests.exceptions.RequestException as e:
        print(f"Error during Health Check: {e}")

    # Test 2: Get all classes
    try:
        response = requests.get(f"{API_URL}/classes")
        print("Classes:", response.json())
    except requests.exceptions.RequestException as e:
        print(f"Error getting Classes: {e}")

    # Test 3: Make prediction
    data = {"text": "A tomato leaf with dark brown lesions and spots"}
    try:
        response = requests.post(f"{API_URL}/predict/text", data=data)
        result = response.json()

        # Check if 'detail' is in the response (indicating an error from FastAPI)
        if 'detail' in result:
            print(f"Error during Prediction: {result['detail']}")
        else:
            print(f"Predicted: {result['prediction']}")
            print(f"Confidence: {result['confidence']:.2%}")

            print("\nTop 3 predictions:")
            for disease, prob in list(result['top_5_predictions'].items())[:3]:
                print(f"  {disease}: {prob:.2%}") # Corrected f-string syntax

    except requests.exceptions.RequestException as e:
        print(f"Error making Prediction request: {e}")
    except KeyError as e:
        print(f"Error processing Prediction response: Missing key {e}")

Using Ngrok Public URL: https://overtediously-uncravatted-ahmed.ngrok-free.dev
INFO:     34.125.134.236:0 - "GET /health-check HTTP/1.1" 404 Not Found
Health Check: {'detail': 'Not Found'}
INFO:     34.125.134.236:0 - "GET /classes HTTP/1.1" 200 OK
Classes: {'classes': ['Pepper bell Bacterial spot', 'Pepper bell healthy', 'Potato Early blight', 'Potato Late blight', 'Potato healthy', 'Tomato Bacterial spot', 'Tomato Early blight', 'Tomato Late blight', 'Tomato Leaf Mold', 'Tomato Septoria leaf spot', 'Tomato Spider mites Two spotted spider mite', 'Tomato Target Spot', 'Tomato YellowLeaf Curl Virus', 'Tomato healthy', 'Tomato mosaic virus'], 'num_classes': 15}
INFO:     34.125.134.236:0 - "POST /predict/text HTTP/1.1" 404 Not Found
Error during Prediction: Not Found


In [6]:
from pyngrok import ngrok

# List all active tunnels
tunnels = ngrok.get_tunnels()
print("Active tunnels:", tunnels)

# Kill all tunnels
ngrok.kill()  # This will terminate all active tunnels

print("✅ All ngrok tunnels have been closed.")


Active tunnels: [<NgrokTunnel: "https://overtediously-uncravatted-ahmed.ngrok-free.dev" -> "http://localhost:8000">]
✅ All ngrok tunnels have been closed.


In [7]:
# ============================================
# RUN THIS IN GOOGLE COLAB
# ============================================

!pip install -q fastapi uvicorn nest-asyncio pyngrok transformers torch pillow

import os
import json
import logging
from typing import Optional, Dict, List

import torch
import pickle
from fastapi import FastAPI, Form, HTTPException
from fastapi.responses import JSONResponse
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import nest_asyncio
from pyngrok import ngrok
import uvicorn

# Enable nested asyncio for Colab
nest_asyncio.apply()

# Configuration
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Create FastAPI app
app = FastAPI(
    title="🌿 Plant Disease Detection API",
    description="Plant disease classification using DistilBERT",
    version="1.0.0",
    docs_url="/docs",
    redoc_url="/redoc"
)

# Paths
TEXT_MODEL_PATH = "./plant_disease_model"
LABEL_ENCODER_PATH = "./plant_disease_model/label_encoder.pkl"

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Using device: {device}")

# Global variables
text_model = None
tokenizer = None
label_encoder = None
text_class_labels = None

# Load Model
try:
    logger.info("Loading model...")
    tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_PATH)
    text_model = AutoModelForSequenceClassification.from_pretrained(TEXT_MODEL_PATH)
    text_model.to(device)
    text_model.eval()

    with open(LABEL_ENCODER_PATH, "rb") as f:
        label_encoder = pickle.load(f)
    text_class_labels = label_encoder.classes_.tolist()

    logger.info(f"✅ Model loaded with {len(text_class_labels)} classes")
except Exception as e:
    logger.exception(f"❌ Error: {e}")

# Helper Functions
def get_label_by_index(labels: Optional[List[str]], idx: int) -> str:
    if labels is None or idx < 0 or idx >= len(labels):
        return str(idx)
    return labels[idx]

def get_top_k_predictions(probs: torch.Tensor, labels: List[str], k: int = 5) -> Dict[str, float]:
    top_k_probs, top_k_indices = torch.topk(probs, k=min(k, len(probs)))
    return {
        get_label_by_index(labels, idx.item()): round(prob.item(), 4)
        for prob, idx in zip(top_k_probs, top_k_indices)
    }

# API Endpoints
@app.get("/")
async def root():
    """Root endpoint with API information."""
    return {
        "message": "🌿 Plant Disease Detection API",
        "status": "running",
        "version": "1.0.0",
        "model": "DistilBERT",
        "num_classes": len(text_class_labels) if text_class_labels else 0
    }

@app.get("/health-check")
def health_check():
    """Health check endpoint."""
    return {
        "status": "ok",
        "message": "API running successfully",
        "model_loaded": text_model is not None,
        "num_classes": len(text_class_labels) if text_class_labels else 0,
        "device": str(device)
    }

@app.get("/classes")
async def get_classes():
    """Get all available disease classes."""
    if text_class_labels is None:
        raise HTTPException(status_code=500, detail="Labels not loaded")
    return JSONResponse({
        "classes": text_class_labels,
        "num_classes": len(text_class_labels),
        "model_type": "DistilBERT"
    })

@app.get("/model-info")
async def model_info():
    """Get detailed model information."""
    return {
        "model_name": "DistilBERT",
        "model_path": TEXT_MODEL_PATH,
        "num_classes": len(text_class_labels) if text_class_labels else 0,
        "device": str(device),
        "model_loaded": text_model is not None
    }

@app.post("/predict/text")
async def predict_text(text: str = Form(...)):
    """
    Predict plant disease from text description.

    Example: "A tomato leaf with dark brown lesions and spots"
    """
    if text_model is None or tokenizer is None:
        raise HTTPException(status_code=500, detail="Model not loaded")
    if not text.strip():
        raise HTTPException(status_code=400, detail="Text required")

    try:
        inputs = tokenizer(text, return_tensors="pt", truncation=True,
                          padding=True, max_length=128)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = text_model(**inputs)
            probs = torch.softmax(outputs.logits, dim=1)[0]
            pred_idx = int(torch.argmax(probs).item())
            confidence = float(probs[pred_idx])

        pred_label = get_label_by_index(text_class_labels, pred_idx)
        top_predictions = get_top_k_predictions(probs, text_class_labels, k=5)

        return JSONResponse({
            "prediction": pred_label,
            "class_index": pred_idx,
            "confidence": round(confidence, 4),
            "top_5_predictions": top_predictions,
            "input_text": text
        })
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Error: {str(e)}")

# ============================================
# NGROK SETUP - THIS IS IMPORTANT FOR COLAB!
# ============================================

# Set your ngrok auth token
ngrok.set_auth_token("34T2QFLGNHYYiQz2T0P9QIEWQ2M_6oGMapkZKhK3oer1UEiJ")

# Start ngrok tunnel
public_url = ngrok.connect(8000)

print("\n" + "="*80)
print("🚀 FASTAPI SERVER IS RUNNING!")
print("="*80)
print(f"🌐 PUBLIC URL: {public_url}")
print(f"📖 INTERACTIVE DOCS: {public_url}/docs")
print(f"📋 REDOC: {public_url}/redoc")
print("="*80)
print(f"\n⚠️  IMPORTANT:")
print(f"   ❌ DON'T use localhost:8000 (won't work in Colab)")
print(f"   ✅ USE this URL instead: {public_url}/docs")
print("="*80)
print(f"\n💡 Copy and open in browser:")
print(f"   {public_url}/docs")
print("="*80 + "\n")

# Run server (Colab-compatible)
import threading

def run_uvicorn():
    uvicorn.run(app, host="0.0.0.0", port=8000)

# Run in a separate thread
thread = threading.Thread(target=run_uvicorn, daemon=True)
thread.start()




🚀 FASTAPI SERVER IS RUNNING!
🌐 PUBLIC URL: NgrokTunnel: "https://overtediously-uncravatted-ahmed.ngrok-free.dev" -> "http://localhost:8000"
📖 INTERACTIVE DOCS: NgrokTunnel: "https://overtediously-uncravatted-ahmed.ngrok-free.dev" -> "http://localhost:8000"/docs
📋 REDOC: NgrokTunnel: "https://overtediously-uncravatted-ahmed.ngrok-free.dev" -> "http://localhost:8000"/redoc

⚠️  IMPORTANT:
   ❌ DON'T use localhost:8000 (won't work in Colab)
   ✅ USE this URL instead: NgrokTunnel: "https://overtediously-uncravatted-ahmed.ngrok-free.dev" -> "http://localhost:8000"/docs

💡 Copy and open in browser:
   NgrokTunnel: "https://overtediously-uncravatted-ahmed.ngrok-free.dev" -> "http://localhost:8000"/docs



INFO:     Started server process [1318]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): [errno 98] address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
